## Different models, Structured Outputs and Guard rails

In [2]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
from pydantic import BaseModel

In [3]:
load_dotenv(override = True)

True

Basically we are definiting the api key, URL end point, set up client to be able to run model with openaichatcompletionmodel which needs model and client

deepseek_api_key = os.getenv('deepseek_api_key')

DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"

deepseek_client = AsyncopenAI(base_url= DEEPSEEK_BASE_URL, api_key=deepseek_api_key)

deepseek_model= openAIchatcompletionModel(model= "deepseek_chat", openai_client = deepeek_client)

In [4]:
openai_api_key = os.getenv('openai_api_key')
google_api_key = os.getenv('google_api_key')
deepseek_api_key = os.getenv('deepseek_api_key')
groq_api_key = os.getenv('groq_api_key')

#if openai_api_key:
#    print(f"OpenAI API key exists and begins {openai_api_key[:8]})
#else:
#    print(f"openai API key not set")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Google API Key not set (and this is optional)
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [5]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

In [7]:
deepseek_client = AsyncOpenAI(base_url=DEEPSEEK_BASE_URL, api_key=deepseek_api_key)
gemini_client= AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=deepseek_api_key)
groq_client= AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

In [8]:
deepseek_model= OpenAIChatCompletionsModel(model= "deepseek_chat", openai_client=deepseek_client)
gemini_model= OpenAIChatCompletionsModel(model= "gemini_2.0-flash", openai_client=gemini_client)
groq_model= OpenAIChatCompletionsModel(model= "llama-3.3-70b-versatile", openai_client=groq_client)


So now agent wants to pass message through specific model

create tool

sales_agent1= Agent (name, instructions, model)

create description what sales agent will put

pass tool and description through sales_agent1

tool1= sales_agent1.as_tool(tool_name, tool_description)


In [9]:

sales_agent1= Agent(name= "Deepseek Sales Agent", instructions=instructions1, model= deepseek_model)
sales_agent2= Agent(name= "Gemini Sales Agent",instructions= instructions2, model = gemini_model)
sales_agent3= Agent(name= "Groq Sales Agent", instructions = instructions3, model = groq_model )


In [10]:
description = "Write a cold sales email."

tool1 = sales_agent1.as_tool(tool_name= "sales_agent1", tool_description= description)
tool2= sales_agent2.as_tool(tool_name= "sales_agent2", tool_description= description)
tool3= sales_agent3.as_tool(tool_name= "sales_agent3", tool_description= description)

Define tool as function to send mail created by agent

Define send grid api, from, to, content, subject
post email and return status

In [11]:
@function_tool
def send_html_email(subject:str, html_body:str)-> Dict[str, str]:
    "Send out an email with given subject and HTML body to all sales prospects"
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("neine112arora@gmail.com")
    to_email= Email ("neine112arora@gmail.com")
    content = Content ("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body = mail)
    return{"status":"success"}
    

sg = sendgrid.SendGridAPIClient(...) — creates a SendGrid client authenticated with an API key pulled from the environment variable SENDGRID_API_KEY.

from_email / to_email — wraps a string address in SendGrid's Email object, which the library needs to build a message.

content — wraps your html_body string, tagging it as "text/html" so SendGrid renders it as HTML rather than plain text.

mail = Mail(...) — bundles sender, recipient, subject, and content into a single Mail object that represents the full message.

sg.client.mail.send.post(...) — makes the actual API call to SendGrid's /mail/send endpoint, passing the mail object's JSON representation as the 

request body.

Return value — reports back {"status": "success"} so the calling agent knows the send worked.

In [12]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer =Agent(name = "Email subject writer", instructions = subject_instructions, model = "gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name= "subject_writer", tool_description="Write a subject for a cold sales email.")

html_converter = Agent(name = "HTML email body converter", instructions = html_instructions, model = "gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name = "html_converter", tool_description="Convert a text body email to an HTML email body")

In [13]:
email_tools = [subject_tool, html_tool, send_html_email]

In [14]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(name = "email_Agent", instructions = instructions, tools = email_tools, model = "gpt-4o-mini", handoff_description="Convert an email to HTML and send it.")

Now tools is list of all model tool, email_tools is subject and html conversion tool , emailer agent is sending tool

In [15]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]

In [16]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent (
    name = "Sales Manager", 
    instructions=sales_manager_instructions, 
    model = "gpt-4o-mini", 
    tools = tools, 
    handoffs = handoffs)


In [17]:
message = "Send out a cold sales email addressed to Dear CEO from Alice"

In [19]:
with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

Creating Guardrails 

1. created class Namecheckoutput with BaseModel 
2. Created guardrail agent with output_type as class created
3. created function (guardrail_agaist_name) with input guardrail_agent,  

NameCheckOutput is a Pydantic model — it forces the LLM's response into a strict, predictable shape instead of free-form text. Instead of getting back "yes, the name is John" as a string you'd have to parse, you get a Python object with .is_name_in_message (a real bool) and .name (a real str).

output_type=NameCheckOutput is what tells the Agents SDK to enforce that shape — under the hood it uses structured output / tool-calling to make the model's response conform to the schema.

model="gpt-4o-mini" — a small, cheap, fast model is a sensible choice here. This is a narrow classification task, not something that needs a frontier model.

This agent is never directly user-facing. It exists purely to be called by the guardrail function below, as a sub-check.

In [20]:
class NameCheckOutput(BaseModel):
    is_name_in_message:bool
    name:str

guardrail_agent= Agent(
    name= "Name Check", 
    instructions = "Check if the user is including someone's personal name in what they want you to do",
    output_type = NameCheckOutput,
    model = "gpt-4o-mini"
)

 This is the key move: instead of writing custom logic to detect names (regex, NER model, etc.), it just runs another agent as the detection mechanism. guardrail_agent gets the message, returns a NameCheckOutput. 
 
 Passing context=ctx.context threads through the same shared context object, so if your context carries things like a user ID or session state, the sub-agent has access to it too.

In [21]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context = ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info= {"found_name": result.final_output}, tripwire_triggered=is_name_in_message)

This is your main, user-facing agent — the one that actually plans and sends the cold email.

tools=tools — whatever tool functions it can call directly (probably things like drafting subject lines, looking up prospect info, etc., depending on your SDR setup).

handoffs=[emailer_agent] — this agent can hand off the conversation to a separate emailer_agent, presumably the one that actually triggers the send via SendGrid. Keeping "decide what to send" and "actually send it" as separate agents is good practice — it's the same instinct as your earlier guardrail point about code-level guardrails being more reliable than prompt-only ones for irreversible actions.

input_guardrails=[guardrail_against_name] — this is where the guardrail from before gets attached. Now, every time this agent receives input, the SDK will run guardrail_against_name first, before careful_sales_manager does anything.

In [24]:
careful_sale_manager = Agent(
    name = "Sales MAnager", 
    instructions = sales_manager_instructions, 
    tools= tools, 
    handoffs =  [emailer_agent],
    model = "gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
)

with trace("Protected Automated SDR") — wraps the whole run in a named trace span, so in your tracing dashboard (OpenAI's, or wherever you've redirected it per your earlier concern) this run shows up grouped under that label. Useful for debugging multi-agent runs since you can see the guardrail check and the main agent run and any handoff, all nested under one trace.

Runner.run(careful_sales_manager, message) — this is the actual kickoff. Internally, before careful_sales_manager processes message, the SDK:

1. Runs guardrail_against_name(ctx, careful_sales_manager, message)

2. That function internally calls guardrail_agent (the gpt-4o-mini name-checker) on the same message

3. guardrail_agent sees "Send out a cold sales email addressed to Dear CEO from Alice" — and "Alice" is a name — so it should return is_name_in_message=True, name="Alice"

4. tripwire_triggered=True comes back

5. The SDK raises InputGuardrailTripwireTriggered (or equivalent) — and careful_sales_manager never runs at all. No tools called, no handoff to emailer_agent, nothing sent.

In [26]:
message = "Send out a cold sales email addressed to Dear CEO from Manager"

with trace("Protected Automated SDR"):
    result = await Runner.run( careful_sale_manager, message)

In [27]:
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sale_manager, message)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire